<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/milestone_5_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 5 — Content action playbook

This stage turns the current-label study into a transparent research review queue. It uses repeated client-held-out predictions from logistic regression and random forest, then aggregates each page's rank across the held-out folds where it appeared. No row is scored by a model trained on that row.

Stage 4's grouped results favored logistic regression on Precision@50 (0.748 vs. 0.708), while random forest had slightly higher average precision (0.663 vs. 0.654). Because this playbook has a fixed top-50 review capacity, logistic regression is the default queue; both model queues are exported for comparison. Reason codes describe observable feature conditions, not explanations of model causality. Treat every item as a human-review candidate, not an automatic refresh instruction.

In [1]:
# Open from the FlyRank repository, or clone it in Colab.
import os, sys, subprocess
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/flyrank-bih/flyrank-ml-internship-starter'
REPO_DIR = 'flyrank-ml-internship-starter'
CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if IN_COLAB and not CSV_PATH.exists():
    if not Path(REPO_DIR).is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
else:
    while not CSV_PATH.exists() and Path.cwd() != Path('/'):
        os.chdir('..')
        CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError('Expected data/raw/content_refresh_anonymized.csv in the FlyRank repo.')
print('CSV:', CSV_PATH.resolve())

CSV: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(CSV_PATH)
required = ['content_id','client_id','trend_direction','days_since_last_update',
            'content_age_days','impressions_90d','avg_position','ctr']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing required fields: {missing}')
df['trend_direction'] = df['trend_direction'].astype('string').str.strip().str.lower()
df['is_declining_label'] = df['trend_direction'].map(
    {'down':1,'new':0,'up':0,'flat':0,'stable':0}
).astype('Int64')
FEATURES = ['days_since_last_update','content_age_days','impressions_90d','avg_position','ctr']
X = df[FEATURES].apply(pd.to_numeric, errors='coerce').copy()
X['avg_position'] = X['avg_position'].mask(X['avg_position'].eq(0))
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']
FEATURES = list(X.columns)
valid = (df['is_declining_label'].notna() & df['content_id'].notna() & df['client_id'].notna() &
         ~df['content_id'].duplicated(keep=False))
eval_df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = eval_df['is_declining_label'].astype(int)
groups = eval_df['client_id']

def build_model(kind, seed=42):
    steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if kind == 'logistic_regression':
        steps += [('scale', StandardScaler()), ('model', LogisticRegression(class_weight='balanced',max_iter=1000,random_state=seed))]
    else:
        steps += [('model', RandomForestClassifier(n_estimators=200,max_depth=5,min_samples_leaf=2,
                   class_weight='balanced',random_state=seed,n_jobs=-1))]
    return Pipeline(steps)
print(f'Eligible rows: {len(eval_df):,}; clients: {groups.nunique()}')

Eligible rows: 30,000; clients: 32


## Held-out rank aggregation

For comparability with Stage 3, this uses five 75/25 `GroupShuffleSplit` partitions with `random_state=42`. Each partition scores only clients held out from that model fit. Pages can appear in multiple test partitions, so we aggregate the fraction of appearances in the top 50 and the mean within-fold rank percentile. This is a stable retrospective queue across these five partitions, not a production score for every page.

In [3]:
N_SPLITS, TEST_SIZE, RANDOM_STATE = 5, 0.25, 42
TOP_K = 50
gss = GroupShuffleSplit(n_splits=N_SPLITS,test_size=TEST_SIZE,random_state=RANDOM_STATE)
prediction_rows=[]
for split_id,(train_idx,test_idx) in enumerate(gss.split(X,y,groups),start=1):
    assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])), 'Client leakage across split.'
    for model_name in ['logistic_regression','random_forest']:
        model = build_model(model_name,RANDOM_STATE)
        model.fit(X.iloc[train_idx],y.iloc[train_idx])
        scores = model.predict_proba(X.iloc[test_idx])[:,1]
        fold = eval_df.iloc[test_idx][['content_id','client_id','trend_direction','is_declining_label',
                                       'days_since_last_update','content_age_days','impressions_90d','avg_position','ctr']].copy()
        fold['split'] = split_id
        fold['model'] = model_name
        fold['priority_score'] = scores
        fold['rank'] = fold['priority_score'].rank(method='first',ascending=False).astype(int)
        fold['rank_percentile'] = fold['rank'] / len(fold)
        fold['top_50'] = fold['rank'].le(TOP_K)
        prediction_rows.append(fold)

heldout_predictions = pd.concat(prediction_rows,ignore_index=True)
consensus = (heldout_predictions.groupby(['model','content_id','client_id'],as_index=False)
    .agg(heldout_appearances=('split','nunique'),top_50_count=('top_50','sum'),
         top_50_rate=('top_50','mean'),mean_rank_percentile=('rank_percentile','mean'),
         mean_priority_score=('priority_score','mean'),trend_direction=('trend_direction','first'),
         is_declining_label=('is_declining_label','first'),days_since_last_update=('days_since_last_update','first'),
         content_age_days=('content_age_days','first'),impressions_90d=('impressions_90d','first'),
         avg_position=('avg_position','first'),ctr=('ctr','first')))
consensus['avg_position'] = consensus['avg_position'].mask(consensus['avg_position'].eq(0))
consensus['mean_rank_percentile'] = consensus['mean_rank_percentile'].fillna(1.0)
consensus['top_50_rate'] = consensus['top_50_rate'].fillna(0.0)
print('Candidate held-out page rows by model:')
display(consensus.groupby('model').agg(candidate_pages=('content_id','nunique'),mean_heldout_appearances=('heldout_appearances','mean')).round(2))

Candidate held-out page rows by model:


,candidate_pages,mean_heldout_appearances
model,,
logistic_regression,20012,1.47
random_forest,20012,1.47


## Reasons and editor checkpoints

Reason codes are plain feature flags, not causal explanations. The position danger band (6–22) follows the existing signal-audit hypothesis and should be checked against a site's own context. Missing position is called out as a data-availability issue. The editor should check seasonality, recent SERP changes, indexing/technical health, intent match, content accuracy, and whether the page should remain active before choosing an action.

Never automate bulk deletion, article rewriting, or URL changes from this queue.

In [4]:
high_visibility_threshold = consensus['impressions_90d'].quantile(0.75)
def reason_codes(row):
    reasons=[]
    age = pd.to_numeric(row['days_since_last_update'],errors='coerce')
    pos = pd.to_numeric(row['avg_position'],errors='coerce')
    impressions = pd.to_numeric(row['impressions_90d'],errors='coerce')
    if pd.isna(pos): reasons.append('POSITION_DATA_MISSING_CHECK_GSC')
    elif 6 <= pos <= 22: reasons.append('POSITION_DANGER_BAND_REVIEW_SERP')
    if pd.notna(age) and age > 180 and pd.notna(impressions) and impressions > 0:
        reasons.append('STALE_WITH_TRAFFIC_REVIEW_UPDATE_HISTORY')
    elif pd.notna(age) and age > 180:
        reasons.append('STALE_PAGE_CHECK_VALUE_AND_STATUS')
    if pd.notna(impressions) and impressions >= high_visibility_threshold:
        reasons.append('HIGH_VISIBILITY_VALIDATE_IMPACT')
    return ';'.join(reasons) if reasons else 'MODEL_PRIORITY_REVIEW_CONTEXT_FIRST'

consensus['reason_codes'] = consensus.apply(reason_codes,axis=1)
consensus['review_prompt'] = ('Check seasonality, technical/indexing status, search intent, SERP changes, content accuracy, and page value before action.')
consensus['recommendation_status'] = 'HUMAN_REVIEW_REQUIRED'
consensus['priority_basis'] = 'held-out rank recurrence; not a causal explanation'

def build_queue(model_name):
    q = consensus.query('model == @model_name').copy()
    q = q.sort_values(['top_50_rate','mean_rank_percentile','mean_priority_score'],
                      ascending=[False,True,False]).reset_index(drop=True)
    q.insert(0,'queue_rank',np.arange(1,len(q)+1))
    q['review_tier'] = np.where(q['queue_rank']<=20,'TOP_20_REVIEW',
                         np.where(q['queue_rank']<=50,'NEXT_30_REVIEW','BACKLOG'))
    return q

logistic_queue = build_queue('logistic_regression')
forest_queue = build_queue('random_forest')
display(logistic_queue.head(10)[['queue_rank','content_id','client_id','top_50_rate','mean_rank_percentile','reason_codes','review_tier']])

,queue_rank,content_id,client_id,top_50_rate,mean_rank_percentile,reason_codes,review_tier
0,1,content_24abafed9707,client_8722616204,1.0,0.000100,STALE_WITH_TRAFFIC_REVIEW_UPDATE_HISTORY,TOP_20_REVIEW
1,2,content_9b28cf5ae4f3,client_f369cb89fc,1.0,0.000157,MODEL_PRIORITY_REVIEW_CONTEXT_FIRST,TOP_20_REVIEW
2,3,content_7736e6144f3b,client_624b60c58c,1.0,0.000291,STALE_WITH_TRAFFIC_REVIEW_UPDATE_HISTORY,TOP_20_REVIEW
3,4,content_40e140ba2934,client_8722616204,1.0,0.000299,STALE_WITH_TRAFFIC_REVIEW_UPDATE_HISTORY,TOP_20_REVIEW
4,5,content_b08562686d22,client_f369cb89fc,1.0,0.000315,HIGH_VISIBILITY_VALIDATE_IMPACT,TOP_20_REVIEW
5,6,content_f552433bab3c,client_3fdba35f04,1.0,0.000398,MODEL_PRIORITY_REVIEW_CONTEXT_FIRST,TOP_20_REVIEW
6,7,content_829c68e60d13,client_f369cb89fc,1.0,0.000472,MODEL_PRIORITY_REVIEW_CONTEXT_FIRST,TOP_20_REVIEW
7,8,content_6384168f8647,client_624b60c58c,1.0,0.000582,POSITION_DANGER_BAND_REVIEW_SERP,TOP_20_REVIEW
8,9,content_81fa7fe2098e,client_3fdba35f04,1.0,0.000597,MODEL_PRIORITY_REVIEW_CONTEXT_FIRST,TOP_20_REVIEW
9,10,content_26d48a980581,client_f369cb89fc,1.0,0.000630,MODEL_PRIORITY_REVIEW_CONTEXT_FIRST,TOP_20_REVIEW


## Use limits and monitoring

This queue is for retrospective evaluation against the current label. The CSV does not establish that model features predate the decline label, and repeated held-out appearances are not independent observations. Before using this as a live editorial tool, verify dated feature windows, evaluate on a later time period, and measure realized outcomes. A proposed monitoring trigger such as Precision@50 below 0.60 should be treated as a project threshold to approve and validate, not an established safety guarantee.

In [5]:
OUTPUT_DIR = Path('work/outputs/milestone_5_action_playbook')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
EXPORT_COLUMNS = ['queue_rank','review_tier','content_id','client_id','top_50_count','heldout_appearances',
    'top_50_rate','mean_rank_percentile','mean_priority_score','reason_codes','review_prompt',
    'recommendation_status','priority_basis','days_since_last_update','content_age_days',
    'impressions_90d','avg_position','ctr']
logistic_queue[EXPORT_COLUMNS].to_csv(OUTPUT_DIR/'logistic_regression_review_queue.csv',index=False)
forest_queue[EXPORT_COLUMNS].to_csv(OUTPUT_DIR/'random_forest_review_queue.csv',index=False)
heldout_predictions.to_csv(OUTPUT_DIR/'heldout_fold_predictions.csv',index=False)
contract = {
    'default_queue':'logistic regression, selected for higher grouped Precision@50 in Milestone 4',
    'comparison_queue':'random forest also exported; it had slightly higher grouped average precision in Milestone 4',
    'aggregation':'five repeated GroupShuffleSplit held-out rankings; top-50 frequency then mean rank percentile',
    'unit':'one unique content_id within client_id',
    'reason_codes':'transparent feature flags; not causal or model explanations',
    'intended_use':'retrospective human review of current-label prioritization candidates',
    'not_valid_for':['automatic editing or deletion','future decline prediction','production use before temporal validation'],
}
(OUTPUT_DIR/'action_playbook_contract.json').write_text(json.dumps(contract,indent=2)+'\n',encoding='utf-8')
print('Saved review queues to',OUTPUT_DIR.resolve())

Saved review queues to /content/flyrank-ml-internship-starter/work/outputs/milestone_5_action_playbook
